# Supplementary Table Variable Dictionary Builder

This notebook helps you build, edit, and validate variable metadata dictionaries for Supplementary Tables 3–6. Once finalized, these dictionaries will be used for automated, informative reporting in your main pipeline notebook.

## 1. Set Paths and Load Source Tables

This section sets up paths for raw input files and dictionary outputs, and loads all source tables into pandas DataFrames for further analysis.

In [33]:
from pathlib import Path
import pandas as pd
import yaml
import json

# Set up paths
base_dir = Path.cwd().resolve()
raw_dir = base_dir / "raw_files"
dict_dir = base_dir / "files"  # or Path("config") for config storage
dict_dir.mkdir(exist_ok=True)

# Helper: load any tabular file

def load_table(path):
    ext = path.suffix.lower()
    if ext == ".tsv":
        return pd.read_csv(path, sep="\t")
    if ext == ".csv":
        return pd.read_csv(path)
    if ext in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if ext == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file format: {path}")

# Load all files in raw_files/
raw_files = sorted([p for p in raw_dir.iterdir() if p.is_file()])
tables = {p.stem: load_table(p) for p in raw_files}

print(f"Loaded tables: {list(tables.keys())}")

Loaded tables: ['S3_serratus', 'S4_253SRR_metadata', 'S5_510discovered_contigs', 'S6_reference_database']


## 2. Extract Candidate Variables Across Files

This section summarizes columns, dtypes, and example values for each table, and computes the union/intersection of all variables.

In [34]:
from collections import defaultdict

# Collect per-table variable info
table_vars = {}
for name, df in tables.items():
    table_vars[name] = {
        'columns': list(df.columns),
        'dtypes': {col: str(df[col].dtype) for col in df.columns},
        'examples': {col: df[col].dropna().unique()[:3].tolist() for col in df.columns}
    }

# Compute global union/intersection
all_columns = [set(info['columns']) for info in table_vars.values()]
union_cols = set.union(*all_columns)
intersect_cols = set.intersection(*all_columns)

print("Union of all columns:", union_cols)
print("Intersection of all columns:", intersect_cols)
print("Per-table variable summary:")
for name, info in table_vars.items():
    print(f"\n{name}:")
    for col in info['columns']:
        print(f"  {col} | dtype: {info['dtypes'][col]} | examples: {info['examples'][col]}")

Union of all columns: {'arrayexpress-species', 'sample_title', 'geographic_location', 'cluster_representative', 'total_spots', 'isolation_source', 'library_strategy', 'ena_fastq_http', 'identified_by', 'organism part', 'orf1_tree_representative', 'orf2_rdrp_partial', 'submitter id', 'sample name', 'sample description', 'accession', 'evalue', 'insdc first public', 'ecotype', 'isolation source', 'orf2_start', 'growth condition', 'run_total_spots', 'ena-last-update', 'insdc status', 'collection date', 'orf2_rdrp_length', 'sra_sequence', 'sequencing method', 'first_blastx_hit_alignment_length_aa', 'file', 'aws_free_egress', 'public_filename', 'samp_size', 'environment (feature)', 'env_material', 'common name', 'training', 'tissue', 'max_filter_size', 'sequencing_machine', 'depth', 'experiment_title', 'orf3_tree_representative', 'total_size', 'cell_line', 'study_title', 'sample_timepoint', 'ena_fastq_http_1', 'env_feature', 'geographic location (longitude)', 'orf3_stop', 'min_filter_size', 

## 3. Define Constants, Hints, and Helper Functions

This section defines all the inference rules, hints, and utility functions for inferring variable descriptions, units, and constraints.

In [35]:
import re
import xml.etree.ElementTree as ET

UNCLEAR = "UNCLEAR - human review required"

EXACT_DESC_HINTS = {
    "palm_id": "Palmprint cluster identifier",
    "sotu": "Sequence-based operational taxonomic unit identifier",
    "qseqid": "Query sequence identifier",
    "pident": "Percent identity of the alignment",
    "evalue": "Alignment expectation value",
    "biosample_id": "BioSample accession identifier",
    "run_id": "Sequencing run accession",
    "run_accession": "Sequencing run accession",
    "sample_accession": "Sample accession identifier",
    "sample_title": "Sample title",
    "study_accession": "Study accession identifier",
    "experiment_accession": "Experiment accession identifier",
    "bioproject": "BioProject accession identifier",
    "public_sratoolkit": "Flag indicating whether the file is accessible via SRA Toolkit",
    "gcp_url": "Google Cloud Storage URL",
    "gcp_free_egress": "Google Cloud free-egress region or policy",
    "gcp_access_type": "Google Cloud access type",
    "aws_url": "Amazon Web Services URL",
    "aws_free_egress": "Amazon Web Services free-egress region or policy",
    "aws_access_type": "Amazon Web Services access type",
    "ncbi_url": "NCBI download URL",
    "ncbi_free_egress": "NCBI free-egress policy",
    "ncbi_access_type": "NCBI access type",
    "ebi_url": "EBI download URL",
    "ebi_free_egress": "EBI free-egress policy",
    "ebi_access_type": "EBI access type",
    "lat_lon": "Latitude/longitude coordinate pair",
    "isolate": "Isolate identifier or isolate name",
    "isolation_source": "Material or environment from which the sample was isolated",
    "isolation source": "Material or environment from which the sample was isolated",
    "collection_date": "Collection date",
    "collection date": "Collection date",
    "scientific_name": "Scientific name",
    "common name": "Common name",
    "sample_name": "Sample name",
    "sample name": "Sample name",
    "cell_type": "Cell type",
    "cell type": "Cell type",
    "RNA": "Ribonucleic acid",
    "sampling_prob": "Sampling probability used during training set construction",
    "contig_length": "Contig length",
}

TOKEN_DESC_RULES = [
    (("run", "total", "spots"), "Total number of spots for the sequencing run"),
    (("run", "total", "bases"), "Total number of bases for the sequencing run"),
    (("run", "alias"), "Submitter-provided run alias"),
    (("public", "filename"), "Publicly available file name"),
    (("public", "size"), "Size of the public file"),
    (("public", "date"), "Public release date"),
    (("public", "md5"), "MD5 checksum for the public file"),
    (("public", "version"), "Version number of the public file"),
    (("public", "semantic", "name"), "Public file format label"),
    (("public", "supertype"), "Public file distribution class"),
    (("experiment", "title"), "Experiment title"),
    (("experiment", "desc"), "Experiment description"),
    (("study", "title"), "Study title"),
    (("organism", "taxid"), "NCBI taxonomic identifier for the organism"),
    (("organism", "name"), "Organism name"),
    (("library", "layout"), "Library layout"),
    (("library", "strategy"), "Library strategy"),
    (("library", "source"), "Library source"),
    (("library", "selection"), "Library selection method"),
    (("instrument", "model", "desc"), "Instrument model description"),
    (("instrument", "model"), "Instrument model"),
    (("host", "subject", "id"), "Host subject identifier"),
    (("source", "sample", "category"), "Broad source sample category"),
    (("source", "sample", "subcategory"), "Source sample subcategory"),
    (("ground", "truth", "category"), "Ground-truth category label"),
    (("model", "prediction", "probabiility"), "Model prediction probability"),
    (("model", "prediction", "probability"), "Model prediction probability"),
    (("model", "prediction"), "Model prediction label"),
    (("first", "blastx", "hit", "e", "value"), "E-value of the top BLASTX hit"),
    (("first", "blastx", "hit", "bit", "score"), "Bit score of the top BLASTX hit"),
    (("first", "blastx", "hit", "identity", "percent"), "Percent identity of the top BLASTX hit"),
    (("first", "blastx", "hit", "alignment", "length", "aa"), "Amino-acid alignment length of the top BLASTX hit"),
    (("first", "diamond", "blastx", "hit", "identity", "percent"), "Percent identity of the top DIAMOND BLASTX hit"),
    (("first", "diamond", "blastx", "hit", "alignment", "length", "aa"), "Amino-acid alignment length of the top DIAMOND BLASTX hit"),
    (("genbank", "accession", "number"), "GenBank accession number"),
    (("geo", "loc", "name"), "Geographic location name"),
    (("geographic", "location", "latitude"), "Sampling latitude"),
    (("geographic", "location", "longitude"), "Sampling longitude"),
    (("geographic", "location", "country", "and", "or", "sea"), "Geographic location: country and/or sea"),
    (("sequencing", "method"), "Sequencing method"),
    (("sequencing", "machine"), "Sequencing machine"),
    (("collection", "date"), "Collection date"),
    (("ena", "first", "public"), "ENA first public release date"),
    (("ena", "last", "update"), "ENA last update date"),
    (("insdc", "first", "public"), "INSDC first public release date"),
    (("insdc", "last", "update"), "INSDC last update date"),
    (("orf1", "ref"), "Reference identifier for ORF1"),
    (("orf2", "ref"), "Reference identifier for ORF2"),
    (("cp", "ref"), "Reference identifier for coat protein"),
    (("orf1", "record", "id"), "Record identifier for ORF1"),
    (("orf2", "record", "id"), "Record identifier for ORF2"),
    (("cp", "record", "id"), "Record identifier for coat protein"),
]

SINGLE_TOKEN_DESC_HINTS = {
    "biosample": "BioSample accession identifier",
    "sample": "Sample identifier",
    "run": "Sequencing run accession",
    "sra": "SRA accession",
    "file": "Source file name or path",
    "path": "Source file path",
    "date": "Collection or processing date",
    "year": "Collection or processing year",
    "country": "Country of origin",
    "location": "Sampling location",
    "lat": "Latitude",
    "latitude": "Latitude",
    "lng": "Longitude",
    "lon": "Longitude",
    "longitude": "Longitude",
    "host": "Host organism",
    "virus": "Virus or viral taxon",
    "taxid": "Taxonomic identifier",
    "tax": "Taxonomic annotation",
    "family": "Taxonomic family",
    "genus": "Taxonomic genus",
    "species": "Taxonomic species",
    "lineage": "Taxonomic lineage",
    "contig": "Contig identifier",
    "length": "Sequence length",
    "start": "Start position",
    "stop": "Stop position",
    "end": "End position",
    "coverage": "Read coverage",
    "depth": "Read depth",
    "count": "Count",
    "abundance": "Relative abundance",
    "identity": "Sequence identity",
    "similarity": "Sequence similarity",
    "bitscore": "Alignment bit score",
    "score": "Alignment score",
    "strand": "Strand orientation",
    "platform": "Sequencing platform",
    "training": "Training-set membership flag",
    "accession": "Accession identifier",
    "nickname": "Nickname",
    "sequence": "Nucleotide sequence",
    "assembler": "Assembler name",
    "notes": "Free-text notes",
    "submitter": "Submitter name",
    "type": "Record type",
}

LOW_CONFIDENCE_TOKENS = {
    "otu", "sotu", "qseqid", "pident", "rna", "cp", "orf1", "orf2", "orf3", "orf4",
    "rdrp", "mp", "conduc", "elev", "temp", "cultivar", "ecotype", "phenotype",
}

EXACT_BINARY_FIELDS = {
    "public_sratoolkit",
    "training",
    "cluster_representative",
    "known_or_potentially_novel_tobamovirus",
}

BINARY_HINT_TOKENS = {
    "flag", "complete", "partial", "representative", "training", "novel", "toolkit", "prediction"
}

# ============================================================================
# Helper functions for normalization and token matching
# ============================================================================

def to_yaml_safe(value):
    """Convert pandas/numpy types to YAML-safe Python types."""
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    return value


def normalize_column_name(col_name: str) -> tuple[str, list[str]]:
    """Convert column name to normalized form and token list."""
    spaced = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", str(col_name))
    normalized = re.sub(r"[^A-Za-z0-9]+", " ", spaced).strip().lower()
    tokens = [token for token in normalized.split() if token]
    return normalized, tokens


def normalize_ncbi_key(name: str) -> str:
    """Normalize NCBI attribute names for matching."""
    normalized = re.sub(r"[^A-Za-z0-9]+", " ", str(name)).strip().lower()
    return normalized


def humanize_column_name(col_name: str) -> str:
    """Convert column name to human-readable form."""
    normalized, _ = normalize_column_name(col_name)
    return normalized.capitalize() if normalized else str(col_name)


def has_all_tokens(tokens: list[str], required: tuple[str, ...]) -> bool:
    """Check if all required tokens are present."""
    token_set = set(tokens)
    return all(token in token_set for token in required)


def is_binary_like_values(series: pd.Series) -> bool:
    """Check if series contains only binary-like values."""
    s = series.dropna()
    if s.empty:
        return False
    if pd.api.types.is_bool_dtype(s):
        return True
    unique_vals = {to_yaml_safe(v) for v in s.unique().tolist()}
    return unique_vals.issubset({0, 1, 0.0, 1.0, True, False})


def should_treat_as_binary(col_name: str, series: pd.Series) -> bool:
    """Determine if column should be treated as binary."""
    normalized, tokens = normalize_column_name(col_name)
    if not is_binary_like_values(series):
        return False
    if pd.api.types.is_bool_dtype(series.dropna()):
        return True
    if col_name in EXACT_BINARY_FIELDS or normalized in EXACT_BINARY_FIELDS:
        return True
    return any(token in BINARY_HINT_TOKENS for token in tokens)


def extract_unit_from_format(format_str: str) -> str:
    """Extract unit information from NCBI format string."""
    if not format_str:
        return None
    fmt_lower = format_str.lower()
    if "meter" in fmt_lower:
        return "meters"
    if "degree" in fmt_lower:
        return "degrees"
    if "percent" in fmt_lower or "%" in fmt_lower:
        return "%"
    if "days" in fmt_lower or "day" in fmt_lower:
        return "days"
    return None


def is_categorical_format(format_str: str) -> bool:
    """Check if NCBI format indicates categorical data."""
    if not format_str:
        return False
    fmt_lower = format_str.lower()
    return any(term in fmt_lower for term in ["text", "term", "value", "enum", "one of"])


print("✓ Constants and helper functions loaded")

✓ Constants and helper functions loaded


In [36]:
# ============================================================================
# Load and parse NCBI BioSample attribute definitions
# ============================================================================

ncbi_attributes = {}

# Try to load NCBI XML file (if available in the system)
# Check in order: local project files, system locations, home directory
ncbi_xml_candidates = [
    Path.cwd() / "files" / "ncbi_biosample_attributes.xml",  # Local files directory
    base_dir / "files" / "ncbi_biosample_attributes.xml",  # Relative to script base dir
    Path("/usr/local/ncbi/biosample/biosample_attributes.xml"),  # System location
    Path.home() / ".ncbi" / "biosample_attributes.xml",  # Home directory
]

ncbi_xml_path = None
for candidate in ncbi_xml_candidates:
    if candidate.exists():
        ncbi_xml_path = candidate
        break

if ncbi_xml_path:
    try:
        tree = ET.parse(ncbi_xml_path)
        root = tree.getroot()
        
        for attr in root.findall("Attribute"):
            name_elem = attr.find("Name")
            harm_name_elem = attr.find("HarmonizedName")
            desc_elem = attr.find("Description")
            format_elem = attr.find("Format")
            
            if name_elem is None:
                continue

            name_key = normalize_ncbi_key(name_elem.text)
            description = desc_elem.text if desc_elem is not None else None
            format_str = format_elem.text if format_elem is not None else None
            info = {
                "description": description,
                "format": format_str,
                "name": name_elem.text,
                "harmonized_name": harm_name_elem.text if harm_name_elem is not None else None,
            }
            
            ncbi_attributes[name_key] = info
            if harm_name_elem is not None and harm_name_elem.text:
                harm_name_key = normalize_ncbi_key(harm_name_elem.text)
                ncbi_attributes[harm_name_key] = info
            
            for syn_elem in attr.findall("Synonym"):
                if syn_elem.text:
                    syn_key = normalize_ncbi_key(syn_elem.text)
                    ncbi_attributes[syn_key] = info
        
        print(f"✓ Loaded {len(ncbi_attributes)} NCBI BioSample attribute definitions")
    except Exception as e:
        print(f"⚠ Error loading NCBI XML: {e}")
        print("  Proceeding without NCBI attribute definitions")
else:
    print(f"ℹ NCBI XML file not found at {ncbi_xml_path}")
    print("  Proceeding without NCBI attribute definitions")

✓ Loaded 1751 NCBI BioSample attribute definitions


In [37]:
def infer_description(col_name: str) -> tuple[str, str, str]:
    """
    Infer variable description using hierarchical rules:
    1. NCBI BioSample attributes (if available)
    2. Exact name hints
    3. Token-based rules
    4. Single-token hints
    5. Fallback to unclear
    
    Returns: (description, review_status, inference_method)
    """
    normalized, tokens = normalize_column_name(col_name)

    # First priority: check NCBI attributes
    if ncbi_attributes:
        ncbi_lookup_key = normalize_ncbi_key(col_name)
        normalized_key = normalize_ncbi_key(normalized)
        
        for key in [ncbi_lookup_key, normalized_key]:
            if key in ncbi_attributes and ncbi_attributes[key].get("description"):
                desc = ncbi_attributes[key]["description"]
                return desc, "auto_inferred", "NCBI BioSample attribute"

    # Second priority: exact hints
    if col_name in EXACT_DESC_HINTS:
        return EXACT_DESC_HINTS[col_name], "auto_inferred", "exact-name rule"
    if normalized in EXACT_DESC_HINTS:
        return EXACT_DESC_HINTS[normalized], "auto_inferred", "normalized exact-name rule"

    # Third priority: token rules
    for required_tokens, description in TOKEN_DESC_RULES:
        if has_all_tokens(tokens, required_tokens):
            return description, "auto_inferred", f"token rule: {'+'.join(required_tokens)}"

    # Fourth priority: single-token hints
    for token in tokens:
        if token in SINGLE_TOKEN_DESC_HINTS:
            return SINGLE_TOKEN_DESC_HINTS[token], "review_recommended", f"single-token rule: {token}"

    # Last resort
    if any(token in LOW_CONFIDENCE_TOKENS for token in tokens):
        return UNCLEAR, "unclear", f"unresolved abbreviation in column name: {col_name}"

    return UNCLEAR, "unclear", f"no confident inference rule matched for: {col_name}"


def infer_unit(col_name: str, dtype_str: str, series: pd.Series, review_status: str) -> tuple[str, str]:
    """
    Infer unit of measurement using hierarchical rules:
    1. NCBI format definitions (if available)
    2. Token-based rules
    3. Binary/categorical detection
    4. Numeric bounds
    5. Date parsing
    6. Fallback to generic
    
    Returns: (unit, inference_method)
    """
    normalized, tokens = normalize_column_name(col_name)
    token_set = set(tokens)

    if series.dropna().empty:
        return UNCLEAR, "column has no non-null values"
    
    # First priority: check NCBI format
    if ncbi_attributes:
        ncbi_lookup_key = normalize_ncbi_key(col_name)
        normalized_key = normalize_ncbi_key(normalized)
        
        for key in [ncbi_lookup_key, normalized_key]:
            if key in ncbi_attributes and ncbi_attributes[key].get("format"):
                fmt = ncbi_attributes[key]["format"]
                # Extract unit from format string
                extracted_unit = extract_unit_from_format(fmt)
                if extracted_unit and extracted_unit.lower() not in ['text', 'term', 'value']:
                    return extracted_unit, f"NCBI format: {fmt}"
                # Check if it's categorical
                if is_categorical_format(fmt):
                    return "n/a", f"NCBI categorical: {fmt[:50]}..."

    if {"lat", "latitude"} & token_set or has_all_tokens(tokens, ("geographic", "location", "latitude")):
        return "degrees", "coordinate rule"
    if {"lng", "lon", "longitude"} & token_set or has_all_tokens(tokens, ("geographic", "location", "longitude")):
        return "degrees", "coordinate rule"
    if has_all_tokens(tokens, ("lat", "lon")) or has_all_tokens(tokens, ("latitude", "longitude")):
        return "degrees", "coordinate-pair rule"
    if {"length", "start", "stop", "end", "position"} & token_set:
        return "bp", "sequence-coordinate rule"
    if {"identity", "similarity", "percent"} & token_set:
        return "%", "percentage rule"
    if token_set == {"gc"} or has_all_tokens(tokens, ("gc", "content")):
        return "%", "GC-content rule"
    if "coverage" in token_set:
        return "x", "coverage rule"
    if "date" in token_set:
        return "ISO-8601", "date rule"
    if should_treat_as_binary(col_name, series):
        return "unitless", "binary flag rule"
    if "bool" in dtype_str or "int" in dtype_str or "float" in dtype_str:
        if review_status == "unclear":
            return UNCLEAR, "numeric column without confident semantic match"
        return "unitless", "generic numeric rule"
    return "n/a", "default text rule"


def infer_allowed_range(col_name: str, series: pd.Series) -> tuple[str, str]:
    """
    Infer allowed range of values based on column name and data.
    
    Returns: (allowed_range, inference_method)
    """
    normalized, tokens = normalize_column_name(col_name)
    token_set = set(tokens)
    s = series.dropna()

    if s.empty:
        return UNCLEAR, "column has no non-null values"
    if has_all_tokens(tokens, ("lat", "lon")) or has_all_tokens(tokens, ("latitude", "longitude")):
        return "formatted latitude/longitude pair", "coordinate-pair rule"
    if {"lat", "latitude"} & token_set or has_all_tokens(tokens, ("geographic", "location", "latitude")):
        return "[-90, 90]", "latitude bounds"
    if {"lng", "lon", "longitude"} & token_set or has_all_tokens(tokens, ("geographic", "location", "longitude")):
        return "[-180, 180]", "longitude bounds"
    if should_treat_as_binary(col_name, s):
        return "{0, 1}", "binary flag rule"
    if pd.api.types.is_numeric_dtype(s):
        min_v = float(s.min())
        max_v = float(s.max())
        if min_v >= 0 and max_v <= 1:
            return "[0, 1]", "bounded numeric rule"
        return f"[{min_v:.6g}, {max_v:.6g}]", "numeric min/max rule"

    if "date" in token_set:
        parsed = pd.to_datetime(s, errors="coerce", format="mixed", utc=True)
        parsed = parsed.dropna()
        if not parsed.empty:
            return f"[{parsed.min().date()}, {parsed.max().date()}]", "date parsing rule"

    as_text = s.astype(str)
    unique_vals = sorted(as_text.unique())
    if len(unique_vals) <= 10:
        shown = ", ".join(repr(v) for v in unique_vals)
        return "{" + shown + "}", "small categorical rule"

    return f"free text / categorical ({len(unique_vals)} unique values)", "categorical summary rule"


def build_notes(review_status: str, description_reason: str, unit_reason: str, range_reason: str) -> str:
    """Build comprehensive notes about inference methods."""
    parts = [f"description: {description_reason}", f"unit: {unit_reason}", f"allowed_range: {range_reason}"]
    if review_status != "auto_inferred":
        parts.append("review this field before final reporting")
    return "; ".join(parts)


print("✓ Inference functions loaded")

✓ Inference functions loaded


## 4. Inference Functions for Variable Metadata

This section defines functions to infer descriptions, units, and allowed ranges for variables based on column names, data types, and NCBI attribute definitions.

In [38]:
# ============================================================================
# Build and save metadata dictionaries
# ============================================================================

def build_metadata_template(table_name, info, df):
    """Build complete metadata dictionary for a single table."""
    meta = {}
    for col in info["columns"]:
        dtype = info["dtypes"][col]
        description, review_status, description_reason = infer_description(col)
        unit, unit_reason = infer_unit(col, dtype, df[col], review_status)
        allowed_range, range_reason = infer_allowed_range(col, df[col])

        if description == UNCLEAR or unit == UNCLEAR or allowed_range == UNCLEAR:
            review_status = "unclear"

        meta[col] = {
            "description": description,
            "unit": unit,
            "allowed_range": allowed_range,
            "expected_dtype": dtype,
            "examples": [to_yaml_safe(v) for v in info["examples"][col]],
            "source_files": [table_name],
            "review_status": review_status,
            "notes": build_notes(review_status, description_reason, unit_reason, range_reason),
        }
    return meta


# Build metadata dictionaries for all tables
metadata_dicts = {
    name: build_metadata_template(name, info, tables[name])
    for name, info in table_vars.items()
}

# Save separate draft dictionaries and editable review copies without overwriting final files
for table_name, meta in metadata_dicts.items():
    draft_path = dict_dir / f"{table_name}_variable_dictionary.draft.yaml"
    review_path = dict_dir / f"{table_name}_variable_dictionary.review_copy.yaml"

    with draft_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(meta, f, sort_keys=True, allow_unicode=False)

    with review_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(meta, f, sort_keys=True, allow_unicode=False)

print(f"✓ Built and saved {len(metadata_dicts)} variable dictionaries")

# ============================================================================
# Summary Statistics
# ============================================================================

from collections import Counter

summary_stats = {
    "total_variables": 0,
    "auto_inferred": 0,
    "unclear": 0,
    "review_recommended": 0,
    "review_completed": 0,
    "inference_methods": Counter(),
}

for table_name, meta in metadata_dicts.items():
    for col, info in meta.items():
        summary_stats["total_variables"] += 1
        status = info["review_status"]
        summary_stats[status] = summary_stats.get(status, 0) + 1
        
        # Extract inference method from notes
        notes = info["notes"]
        if "NCBI" in notes:
            summary_stats["inference_methods"]["NCBI BioSample"] += 1
        elif "auto_inferred" in status:
            if "exact-name" in notes:
                summary_stats["inference_methods"]["Exact name rule"] += 1
            elif "token rule" in notes:
                summary_stats["inference_methods"]["Token rule"] += 1
            else:
                summary_stats["inference_methods"]["Other auto-inferred"] += 1
        else:
            summary_stats["inference_methods"][status] += 1

print("=" * 60)
print("VARIABLE DICTIONARY INFERENCE SUMMARY")
print("=" * 60)
print(f"\nTotal variables processed: {summary_stats['total_variables']}")
print(f"  ✓ Auto-inferred (high confidence): {summary_stats['auto_inferred']}")
print(f"  ~ Review recommended (medium confidence): {summary_stats.get('review_recommended', 0)}")
print(f"  ✓ Review completed (user-validated): {summary_stats.get('review_completed', 0)}")
print(f"  ⚠ Unclear (needs human review): {summary_stats['unclear']}")

print(f"\nInference method breakdown:")
for method, count in summary_stats["inference_methods"].most_common():
    pct = 100 * count / summary_stats['total_variables']
    print(f"  {method:.<35} {count:>4} ({pct:>5.1f}%)")

print("\nVariables resolved by NCBI BioSample attributes:")
ncbi_resolved = [
    (table_name, col, info)
    for table_name, meta in metadata_dicts.items()
    for col, info in meta.items()
    if "NCBI" in info["notes"]
]
print(f"  Total: {len(ncbi_resolved)}")
if len(ncbi_resolved) > 0:
    print(f"\n  Examples:")
    for table_name, col, info in ncbi_resolved[:10]:
        print(f"    • {col:.<30} → {info['description'][:50]}")
    if len(ncbi_resolved) > 10:
        print(f"    ... and {len(ncbi_resolved) - 10} more")

import pprint
print("\n" + "=" * 60)
print("SAMPLE OUTPUT: S3_serratus variable metadata")
print("=" * 60)
if "S3_serratus" in metadata_dicts:
    sample_dict = {k: metadata_dicts["S3_serratus"][k] for k in list(metadata_dicts["S3_serratus"].keys())[:3]}
    pprint.pprint(sample_dict)
else:
    print("(S3_serratus table not found in loaded data)")

print(f"\n✓ All dictionaries saved to: {dict_dir}")

✓ Built and saved 4 variable dictionaries
VARIABLE DICTIONARY INFERENCE SUMMARY

Total variables processed: 249
  ✓ Auto-inferred (high confidence): 135
  ~ Review recommended (medium confidence): 39
  ✓ Review completed (user-validated): 0
  ⚠ Unclear (needs human review): 75

Inference method breakdown:
  NCBI BioSample.....................   65 ( 26.1%)
  unclear............................   62 ( 24.9%)
  Token rule.........................   52 ( 20.9%)
  review_recommended.................   39 ( 15.7%)
  Exact name rule....................   31 ( 12.4%)

Variables resolved by NCBI BioSample attributes:
  Total: 65

  Examples:
    • strain........................ → microbial or eukaryotic strain name
    • isolate....................... → identification or description of the specific indi
    • host.......................... → The natural (as opposed to laboratory) host to the
    • isolation_source.............. → Describes the physical, environmental and/or local
    • collect